# Problem Set 3

In [1]:
import gymnasium as gym
import numpy as np

HIT = 1
STICK = 0

env = gym.make("Blackjack-v1")

In [2]:
print(env.observation_space.spaces[0].n)

32


In [3]:
class MCAgent():
    def __init__(self, alpha=0.1, gamma=1.0):
        self.alpha = alpha
        self.gamma = gamma

        self.states = []
        self.actions = []
        self.rewards = []
    
    def _reset(self):
        self.states = []
        self.actions = []
        self.rewards = []
    
    def act(self, obs):
        raise NotImplementedError
    
    def update(self, obs, action, reward, terminated):
        self.states.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)

        if terminated:
            self._update_values()
            self._reset()
    
    def _update_values(self):
        raise NotImplementedError


## Problem 1c

In [4]:
class FixedPolicyAgent(MCAgent):
    def __init__(self, alpha=0.1, gamma=1, threshold=20):
        super().__init__(alpha, gamma)

        self.V = np.zeros((32,11,2))
        self.threshold = threshold
    
    def act(self, obs):
        if obs[0] >= self.threshold:
            return STICK
        return HIT
        
    def _update_values(self):
        assert len(self.states) == len(self.rewards)

        state_count = {}
        for state in self.states:
            if state in state_count:
                state_count[state] += 1
            else:
                state_count[state] = 1

        T = len(self.states)
        G = 0
        for t in range(T-1, -1, -1):
            G = self.gamma * G + self.rewards[t]

            state = self.states[t]
            state_count[state] -= 1

            # first visit
            if state_count[state] == 0:
                current = self.V[state[0]][state[1]][state[2]]
                self.V[state[0]][state[1]][state[2]] = current + self.alpha * (G - current)


In [5]:
NUM_EPISODES = 500000

agent = FixedPolicyAgent()
for episode in range(NUM_EPISODES):
    obs, _ = env.reset()
    terminated = False
    truncated = False

    while not (terminated or truncated):
        curr_obs = obs

        action = agent.act(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        agent.update(curr_obs, action, reward, terminated or truncated)

In [6]:
import plotly.express as px

no_ace = np.transpose(agent.V)[0]
ace = np.transpose(agent.V)[1]
fig = px.imshow(
    no_ace,
    title="Estimated state values (No usable Ace)",
    labels={
        "x": "Player's current sum",
        "y": "Dealer card"
    }
)
fig.show()
fig.write_image("V_no_ace.png")

In [7]:
fig2 = px.imshow(
    ace,
    title="Estimated state values (Usable Ace)",
    labels={
        "x": "Player's current sum",
        "y": "Dealer card"
    },
)
fig2.show()
fig2.write_image("V_ace.png")

## Problem 1d

In [8]:
class EpsilonSoftAgent(MCAgent):
    def __init__(self, alpha=0.1, gamma=1, epsilon=0.1, seed=None):
        super().__init__(alpha, gamma)

        self.Q = np.zeros((32,11,2, 2))
        self.epsilon = epsilon

        self.rng = np.random.default_rng(seed=seed)
    
    def act(self, obs):
        if self.rng.random() < self.epsilon:
            return self.rng.choice([HIT, STICK])
        
        return np.argmax(self.Q[obs[0]][obs[1]][obs[2]])
        
    def _update_values(self):
        assert len(self.states) == len(self.rewards)
        assert len(self.actions) == len(self.states)

        state_action_count = np.zeros(self.Q.shape)
        for (state, action) in zip(self.states, self.actions):
            state_action_count[state[0]][state[1]][state[2]][action] += 1

        T = len(self.states)
        G = 0
        for t in range(T-1, -1, -1):
            G = self.gamma * G + self.rewards[t]

            state = self.states[t]
            action = self.actions[t]

            state_action_count[state[0], state[1], state[2], action] -= 1

            # first visit
            if state_action_count[state[0], state[1], state[2], action] == 0:
                current = self.Q[state[0]][state[1]][state[2]][action]
                self.Q[state[0]][state[1]][state[2]][action] = current + self.alpha * (G - current)

In [9]:
# NUM_EPISODES = 1000000
NUM_EPISODES = 500000

eps_agent = EpsilonSoftAgent()
for episode in range(NUM_EPISODES):
    obs, _ = env.reset()
    terminated = False
    truncated = False

    while not (terminated or truncated):
        curr_obs = obs

        action = eps_agent.act(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        eps_agent.update(curr_obs, action, reward, terminated or truncated)

In [10]:
policy = np.transpose(np.argmax(eps_agent.Q, axis=3))
policy_text = np.where(policy == 1, "hit", "stick")

fig3 = px.imshow(
    policy[0],
    text_auto=True,
    color_continuous_scale=["white", "lightblue"],
    title="Estimated state values (No usable Ace)",
    labels={
        "x": "Player's current sum",
        "y": "Dealer card"
    },)
fig3.update_traces(text=policy_text[0], texttemplate="%{text}", ygap=1, xgap=1)
fig3.update_layout(coloraxis_showscale=False, plot_bgcolor="lightgray")
fig3.show()
fig3.write_image("policy_no_ace.png", scale=2.0)

In [11]:
fig4 = px.imshow(
    policy[1],
    text_auto=True,
    color_continuous_scale=["white", "lightblue"],
    title="Estimated state values (Usable Ace)",
    labels={
        "x": "Player's current sum",
        "y": "Dealer card"
    },)
fig4.update_traces(text=policy_text[1], texttemplate="%{text}", ygap=1, xgap=1)
fig4.update_layout(coloraxis_showscale=False, plot_bgcolor="lightgray")
fig4.show()
fig4.write_image("policy_ace.png", scale=2.0)

## Problem 2c

In [12]:
ACTION_LEFT = 0
ACTION_RIGHT = 1

STATE_A = 0
STATE_B = 1
STATE_C = 2
STATE_D = 3
STATE_E = 4
STATE_F = 5
STATE_G = 6

class RandomWalkEnv(gym.Env):
    def __init__(self, seed=None):
        self.action_space = gym.spaces.Discrete(2)
        self.observation_space = gym.spaces.Discrete(7)

        self._current_state = STATE_D

        super().reset(seed=seed)
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed, options=options)

        self._current_state = STATE_D

        return self._current_state, {}
    
    def step(self, action):
        terminated = False
        truncated = False

        if action == ACTION_LEFT:
            self._current_state -= 1
        else:
            self._current_state += 1

        if self._current_state == STATE_A:
            terminated = True
            return self._current_state, 0, terminated, truncated, {}
        if self._current_state == STATE_G:
            terminated = True
            return self._current_state, 1, terminated, truncated, {}

        return self._current_state, 0, terminated, truncated, {}

In [13]:
env = RandomWalkEnv()

In [21]:
class TDZeroRandomWalkAgent():
    def __init__(self, alpha=0.1, gamma=1.0):
        self.alpha = alpha
        self.gamma = gamma

        self.V = np.zeros(7)

        self._last_state = None

    def act(self, state):
        self._last_state = state
        return env.action_space.sample()
    
    def update(self, state, reward, _):
        current = self.V[self._last_state]
        self.V[self._last_state] += self.alpha * (reward + (self.gamma * self.V[state]) - current)
    
    def reset(self):
        self._last_state = None

In [22]:
class MCRandomWalkAgent():
    def __init__(self, alpha=0.1, gamma=1.0):
        self.alpha = alpha
        self.gamma = gamma

        self.V = np.zeros(7)

        self._states = []
        self._rewards = []

    def act(self, state):
        self._states.append(state)
        return env.action_space.sample()
    
    def update(self, _, reward, terminated):
        self._rewards.append(reward)

        if terminated:
            assert len(self._states) == len(self._rewards)

            state_count = np.zeros(7)
            for state in self._states:
                state_count[state] += 1

            T = len(self._states)
            G = 0
            for t in range(T-1, -1, -1):
                G = self.gamma * G + self._rewards[t]

                state = self._states[t]
                state_count[state] -= 1

                # first visit
                if state_count[state] == 0:
                    current = self.V[state]
                    self.V[state] = current + self.alpha * (G - current)
        
    def reset(self):
        self._states = []
        self._rewards = []

In [23]:
true_means = np.array([1/6,2/6,3/6,4/6,5/6])

def test_agent(agent_type, alpha=0.1, gamma=1.0, num_episodes=100):
    agent = agent_type(alpha, gamma)
    rmses = []

    for _ in range(num_episodes):
        obs, _ = env.reset()

        terminated = False
        truncated = False
        while not terminated or truncated:
            action = agent.act(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            agent.update(obs, reward, terminated or truncated)
        
        agent.reset()
        rmses.append(np.sqrt(np.mean((agent.V[1:6] - true_means)**2)))
    
    return rmses

In [24]:
rmse_td = test_agent(TDZeroRandomWalkAgent)
print(f"RMSE from TD(0) agent: {rmse_td[-1]}")

rmse_mc = test_agent(MCRandomWalkAgent)
print(f"RMSE from MC agent: {rmse_mc[-1]}")

RMSE from TD(0) agent: 0.034950103104543995
RMSE from MC agent: 0.1730902331666287


## Problem 2d

In [47]:
rmses_td_0_01 = []
for _ in range(50):
    rmses_td_0_01.append(test_agent(TDZeroRandomWalkAgent, 0.01))
rmses_td_0_02 = []
for _ in range(50):
    rmses_td_0_02.append(test_agent(TDZeroRandomWalkAgent, 0.02))
rmses_td_0_04 = []
for _ in range(50):
    rmses_td_0_04.append(test_agent(TDZeroRandomWalkAgent, 0.04))

rmses_td_0_01 = np.mean(np.array(rmses_td_0_01), axis=0)
rmses_td_0_02 = np.mean(np.array(rmses_td_0_02), axis=0)
rmses_td_0_04 = np.mean(np.array(rmses_td_0_04), axis=0)

rmses_mc_0_05 = []
for _ in range(50):
    rmses_mc_0_05.append(test_agent(MCRandomWalkAgent, 0.05))
rmses_mc_0_1 = []
for _ in range(50):
    rmses_mc_0_1.append(test_agent(MCRandomWalkAgent, 0.1))
rmses_mc_0_15 = []
for _ in range(50):
    rmses_mc_0_15.append(test_agent(MCRandomWalkAgent, 0.15))

rmses_mc_0_05 = np.mean(np.array(rmses_mc_0_05), axis=0)
rmses_mc_0_1 = np.mean(np.array(rmses_mc_0_1), axis=0)
rmses_mc_0_15 = np.mean(np.array(rmses_mc_0_15), axis=0)

In [49]:
import pandas as pd

# print(rmses_mc[0])
# print(rmses_td[0])

df = pd.DataFrame(np.array([rmses_td_0_01, rmses_td_0_02, rmses_td_0_04, rmses_mc_0_05, rmses_mc_0_1, rmses_mc_0_15]).transpose(), columns=["TD(0); 0.01", "TD(0); 0.02", "TD(0); 0.04", "MC; 0.05", "MC; 0.1", "MC; 0.15"])
# print(df)
fig5 = px.line(df, x = df.index, y = df.columns, title="RMSE of TD(0) vs Monte Carlo",
        labels={"index": "Episodes", "value": "RMSE", "variable": "Algorithm"})
fig5.show()
fig5.write_image("td_vs_mc.png")
